# Exploração dos dados de gráficos



In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import csv
import matplotlib.pyplot as plt

# Funciona tanto a partir da raiz do projeto quanto da pasta notebooks.
projeto = Path.cwd()
if not (projeto / 'dados_projeto_multi_modais').is_dir():
    projeto = projeto.parent
dados = projeto / 'dados_projeto_multi_modais' / 'dados'
if not dados.is_dir():
    raise FileNotFoundError(f'Pasta de dados não encontrada: {dados}')

indice = 1  # Escolha um número entre 1 e 120.
if not 1 <= indice <= 120:
    raise ValueError('O índice deve estar entre 1 e 120.')

def carregar(tipo, numero=indice):
    caminho = dados / f'{tipo}-{numero:03d}.csv'
    with caminho.open(encoding='utf-8', newline='') as arquivo:
        return list(csv.DictReader(arquivo))


## Inventário e prévia dos dados
Cada arquivo corresponde a um gráfico. A prévia mostra as três primeiras linhas do arquivo escolhido em cada categoria.

In [ ]:
tipos = {'BAR': 'Barras', 'LIN': 'Linhas', 'DIS': 'Dispersão',
         'PIZ': 'Pizza', 'HIS': 'Histograma', 'BOX': 'Boxplot'}
contagem = Counter(p.name.split('-')[0] for p in dados.glob('*.csv'))
for sigla, nome in tipos.items():
    linhas = carregar(sigla)
    print(f'\n{nome} ({sigla}): {contagem[sigla]} arquivos | {len(linhas)} linhas no exemplo')
    for linha in linhas[:3]:
        print(' ', linha)


## Visualizar os seis tipos
O histograma usa as contagens por faixa já presentes no CSV; o boxplot é calculado a partir das observações de cada grupo.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 18), constrained_layout=True)

bar = carregar('BAR')
axes[0, 0].bar([r['categoria'] for r in bar], [float(r['valor']) for r in bar], color='#4776a8')
axes[0, 0].set_ylabel('Valor')
axes[0, 0].tick_params(axis='x', rotation=45)

lin = carregar('LIN')
anos = [int(r['ano']) for r in lin]
for serie in ('Série A', 'Série B', 'Série C'):
    axes[0, 1].plot(anos, [float(r[serie]) for r in lin], marker='.', label=serie)
axes[0, 1].set_xlabel('Ano')
axes[0, 1].set_ylabel('Valor')
axes[0, 1].legend()

dis = carregar('DIS')
axes[1, 0].scatter([float(r['x']) for r in dis], [float(r['y']) for r in dis], alpha=0.75)
axes[1, 0].set_xlabel('x')
axes[1, 0].set_ylabel('y')

piz = carregar('PIZ')
axes[1, 1].pie([float(r['percentual']) for r in piz], labels=[r['categoria'] for r in piz], autopct='%1.1f%%')

his = carregar('HIS')
inicio = [float(r['faixa_inicio']) for r in his]
largura = [float(r['faixa_fim']) - float(r['faixa_inicio']) for r in his]
axes[2, 0].bar(inicio, [int(r['contagem']) for r in his], width=largura, align='edge', edgecolor='white')
axes[2, 0].set_xlabel('Faixa')
axes[2, 0].set_ylabel('Contagem')

box = carregar('BOX')
grupos = defaultdict(list)
for r in box:
    grupos[r['grupo']].append(float(r['valor']))
nomes = sorted(grupos)
axes[2, 1].boxplot([grupos[nome] for nome in nomes])
axes[2, 1].set_xticks(range(1, len(nomes) + 1), nomes, rotation=45)
axes[2, 1].set_ylabel('Valor')

for ax, (sigla, nome) in zip(axes.flat, tipos.items()):
    ax.set_title(f'{nome} — {sigla}-{indice:03d}')
plt.show()


## Gabarito
O arquivo `dados_projeto_multi_modais/gabarito.xlsx` contém perguntas e respostas associadas aos gráficos e permanece preservado. Para abrir os links das imagens, use a cópia idêntica `data/processed/gabarito.xlsx`. Os PNGs estão em `data/processed/graficos/`; consulte `galeria_png.ipynb` para visualizá-los. O manifesto `data/manifests/imagens.json` registra a origem reconstruída e os hashes dos dados. Evite fornecer o gabarito ao modelo durante uma avaliação.